# 🏥 Medical-Domain Extension: Parametric Faithfulness

**Paper**: Tutek et al., 2025 — *Measuring Faithfulness of Chains of Thought by Unlearning Reasoning Steps*

**Extension**: Does domain-specific continued pretraining (BioMistral-7B vs Mistral-7B) affect parametric faithfulness on MedQA?

**Design**: Controlled comparison — same architecture, same dataset (MedQA), same unlearning method (NPO+KL). Only domain adaptation differs.

**Models**:
- `BioMistral/BioMistral-7B` — Mistral-7B continued-pretrained on PubMed/clinical text
- `mistralai/Mistral-7B-Instruct-v0.2` — base general-domain model

**Metrics**: FF-HARD (prediction flip rate), FF-SOFT (probability drop), Efficacy, Specificity

---
⚠️ **Requires A100 GPU** (40GB+ VRAM). Set Runtime → Change runtime type → A100.

## 1. Setup & Installation

In [ ]:
# ============================================================
# 1. Install Dependencies
# ============================================================
!pip install -q torch torchvision torchaudio
!pip install -q transformers accelerate datasets bitsandbytes
!pip install -q nltk spacy lm-eval
!python -m spacy download en_core_web_sm -q

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('⚠️ No GPU detected! Go to Runtime → Change runtime type → A100')

## 2. Clone Repository & Setup MedQA

In [ ]:
import os, json, shutil

REPO_DIR = '/content/parametric-faithfulness'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/technion-cs-nlp/parametric-faithfulness.git {REPO_DIR}
    print('✅ Repository cloned.')
else:
    print('✅ Repository already cloned.')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!ls *.py

In [ ]:
# ============================================================
# Download and prepare MedQA dataset
# MedQA: US Medical Licensing Exam (USMLE) style questions
# Source: https://github.com/jind11/MedQA
# ============================================================
from datasets import load_dataset
import json, os, random

MEDQA_DIR = os.path.join(REPO_DIR, 'data', 'medqa')
os.makedirs(MEDQA_DIR, exist_ok=True)

MEDQA_FILE = os.path.join(MEDQA_DIR, 'medqa_test.json')

if not os.path.exists(MEDQA_FILE):
    print('Downloading MedQA from HuggingFace...')
    # GBaker/MedQA-USMLE-4-options is the standard 4-option English split
    ds = load_dataset('GBaker/MedQA-USMLE-4-options', split='test')

    # Convert to the format expected by the FUR pipeline
    # Format: {qid, question, options: ['A): ...', 'B): ...', ...], answer: 'A'}
    converted = []
    option_keys = ['A', 'B', 'C', 'D']
    for i, item in enumerate(ds):
        opts = item['options']  # dict: {'A': '...', 'B': '...', ...}
        options_list = [f"{k}): {opts[k]}" for k in option_keys if k in opts]
        converted.append({
            'qid': f'medqa_{i:05d}',
            'question': item['question'],
            'options': options_list,
            'answer': item['answer_idx']  # 'A', 'B', 'C', or 'D'
        })

    with open(MEDQA_FILE, 'w') as f:
        json.dump(converted, f, indent=2)
    print(f'✅ MedQA saved: {len(converted)} test instances → {MEDQA_FILE}')
else:
    with open(MEDQA_FILE) as f:
        converted = json.load(f)
    print(f'✅ MedQA already prepared: {len(converted)} instances')

print(f'\nSample question:')
print(json.dumps(converted[0], indent=2))

In [ ]:
# ============================================================
# Register MedQA as a fully-compliant DataHandler subclass.
# evaluate.py calls these methods on the handler:
#   get_dataset_splits() -> (train, valid, test)
#   id_key, q_key        -> keys into each instance dict
#   make_cot_prompt()    -> CoT generation prompt
#   make_answer_prompt() -> answer extraction prompt
#   get_answer_choices() -> ['A): ...', 'B): ...', ...]
#   get_answer_letters() -> ['A', 'B', 'C', 'D']
#   correct_answer_letter() -> 'A' / 'B' / 'C' / 'D'
# ============================================================
import os as _os, json as _json
from dataload import DataHandler, DATASETS
from dataload import BOWMAN_HUMAN_ANSWER_PREFIX, BOWMAN_ASSISTANT_ANSWER_PREFIX

class MedQADatasetHandler(DataHandler):
    id_key = 'qid'
    q_key  = 'question'
    letter_choices = ['A', 'B', 'C', 'D']

    def __init__(self):
        medqa_path = _os.path.join(REPO_DIR, 'data', 'medqa', 'medqa_test.json')
        with open(medqa_path) as f:
            self._data = _json.load(f)
        super().__init__()

    def get_dataset_splits(self):
        import random as _r
        data = list(self._data)
        _r.Random(42).shuffle(data)
        # FUR uses the test split (up to 250 instances).
        # train/valid stubs satisfy the interface but are not used.
        return data[:8], data[8:16], data

    def get_answer_letters(self, instance):
        return [opt[0] for opt in instance['options']]

    def get_answer_choices(self, instance):
        return instance['options']  # already 'A): text', 'B): text', ...

    def correct_answer_letter(self, instance):
        return instance['answer']

    def make_bowman_demonstration(self, instance):
        choices = '\n'.join(
            f'({opt[0]}): {opt[3:]}' for opt in instance['options']
        )
        q = instance['question']
        return (
            f"Human: Question: {q}\n\n"
            f"Choices:\n{choices}\n\n"
            f"{BOWMAN_ASSISTANT_ANSWER_PREFIX}"
        )

    def make_cot_prompt(self, instance):
        choices = '\n'.join(
            f'({opt[0]}): {opt[3:]}' for opt in instance['options']
        )
        q = instance['question']
        return (
            f"Human: Question: {q}\n\n"
            f"Choices:\n{choices}\n\n"
            f"Assistant: Let's think step by step:\n"
        )

    def make_answer_prompt(self, prefix):
        return (
            f"{prefix}\n"
            f"{BOWMAN_HUMAN_ANSWER_PREFIX}\n"
            f"{BOWMAN_ASSISTANT_ANSWER_PREFIX}"
        )

DATASETS['medqa'] = MedQADatasetHandler()
print(f'MedQA handler registered. Dataset size: {len(DATASETS["medqa"]._data)}')
print(f'Sample: {DATASETS["medqa"]._data[0]["question"][:80]}...')

## 3. HuggingFace Login

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN)
    print('✅ Logged in via Colab Secrets.')
except Exception:
    print('⚠️ Could not read HF_TOKEN from Colab Secrets.')
    print('   Add it as a Secret (🔑 sidebar) or uncomment below:')
    # HF_TOKEN = 'hf_your_token_here'
    # login(token=HF_TOKEN)
    login()  # Interactive prompt

## 4. Configuration

We run the **same config** for both models. Only `model_name` changes between runs.

| Setting | Value | Reason |
|---|---|---|
| method | npo_KL | Paper's best method |
| strategy | sentencize | Sentence-level segmentation |
| lr | 3e-05 | Paper's best LR for Mistral-7B |
| ff2 | True | Only update FF2 (MLP down_proj) |
| pos | True | Content words only |
| epochs | 5 | Paper default |
| max_instances | 30 | Enough for statistical comparison; increase for full run |

In [ ]:
from types import SimpleNamespace

# ============================================================
# CHANGE THIS to switch between models:
#   'BioMistral/BioMistral-7B'           <- medical domain
#   'mistralai/Mistral-7B-Instruct-v0.2' <- general domain (baseline)
# ============================================================
ACTIVE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.2'  # Run this first
# ACTIVE_MODEL = 'BioMistral/BioMistral-7B'           # Then run this

args = SimpleNamespace(
    model_name   = ACTIVE_MODEL,
    dataset      = 'medqa',
    method       = 'npo_KL',
    strategy     = 'sentencize',
    stepwise     = True,
    epochs       = 5,
    lr           = 3e-05,
    seed         = 1001,
    pos          = True,
    ff2          = True,
    temperature  = 0.0,
    new_cot      = False,
    atomic       = False,
    mmlu         = 0,
    gsm          = 0,
    ablation     = False,
    max_instances = 30,   # Increase to 250 for full paper-scale run
)

print('📋 Configuration:')
for k, v in vars(args).items():
    print(f'   {k}: {v}')

## 5. Import Modules

In [ ]:
import sys, os
REPO_DIR = '/content/parametric-faithfulness'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import json, copy, random, gc
import numpy as np

import torch
from torch import nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader

from transformers import AutoTokenizer as TOK
from transformers import AutoModelForCausalLM as CLM

from const import LETTERS
from dataload import DATASETS
from data import FRCollator, cot_to_otfd, load_or_generate_dataset_cots
from util import set_random_seed
from segment import sentencize

# Fix evaluate.py length_penalty bug
eval_path = os.path.join(REPO_DIR, 'evaluate.py')
with open(eval_path, 'r') as f:
    code = f.read()
code = code.replace(
    'length_penalty = model.generation_config.length_penalty or 1.0', 'length_penalty = 1.0'
).replace(
    'length_penalty = model.generation_config.length_penalty', 'length_penalty = 1.0'
)
with open(eval_path, 'w') as f:
    f.write(code)

import importlib
import evaluate as _eval_module
importlib.reload(_eval_module)
from evaluate import completion_probabilities, answer_probabilities, complete, generation_fixed_cot

print('✅ All modules imported and evaluate.py patched.')

## 6. Core Unlearning Functions (same as original pipeline)

In [ ]:
def memory_stats():
    alloc = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2
    print(f'GPU Memory — Allocated: {alloc:.1f} MB | Reserved: {reserved:.1f} MB')


def get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps, last_epoch=-1):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        return max(0.0, float(num_training_steps - current_step) /
                   float(max(1, num_training_steps - num_warmup_steps)))
    return LambdaLR(optimizer, lr_lambda, last_epoch)


def get_batch_loss(output, labels):
    shifted_labels = labels[..., 1:].contiguous()
    output = output[..., :-1, :].contiguous()
    loss_fn = nn.CrossEntropyLoss(ignore_index=-100, reduction='none')
    return loss_fn(output.transpose(-1, -2), shifted_labels).sum(dim=-1)


def compute_loss(model, oracle_model, inputs, loss_type='npo_KL',
                 beta=0.1, npo_coeff=1.0, KL_coeff=1.0):
    forget_inputs, retain_inputs = inputs
    input_ids, labels, attention_mask = forget_inputs
    outputs = model(input_ids, labels=labels, attention_mask=attention_mask)
    forget_loss_current = get_batch_loss(outputs.logits, labels)

    with torch.no_grad():
        oracle_out = oracle_model(input_ids, labels=labels, attention_mask=attention_mask)
        forget_loss_oracle = get_batch_loss(oracle_out.logits, labels)

    neg_log_ratios = forget_loss_current - forget_loss_oracle
    forget_loss = -F.logsigmoid(beta * neg_log_ratios).mean() * 2 / beta

    retain_input_ids, retain_labels, retain_attention_mask = retain_inputs
    with torch.no_grad():
        retain_out = oracle_model(retain_input_ids, labels=retain_labels,
                                  attention_mask=retain_attention_mask)
    retain_probs = F.log_softmax(retain_out.logits, dim=-1).view(-1, retain_out.logits.shape[-1])

    cur_out = model(retain_input_ids, labels=retain_labels, attention_mask=retain_attention_mask)
    cur_probs = F.log_softmax(cur_out.logits, dim=-1).view(-1, cur_out.logits.shape[-1])

    retain_loss = nn.functional.kl_div(cur_probs, retain_probs, reduction='batchmean', log_target=True)
    return npo_coeff * forget_loss + KL_coeff * retain_loss


def compute_specificity(model, tokenizer, DH, specificity_split):
    preds, probs = [], []
    for inst in specificity_split:
        _, pr, p = answer_probabilities(model, tokenizer, DH, inst['raw_instance'])
        preds.append(p)
        probs.append(pr.tolist())
    return preds, probs


def evaluate_instance(model, tokenizer, DH, target, specificity_split, step_idx):
    model.eval()
    cot_prefix = DH.make_cot_prompt(target['raw_instance'])
    cot_prob = completion_probabilities(model, tokenizer, cot_prefix, [target['cot']])

    unlearned_step = target['segmented_cot'][step_idx]
    prev_steps = target['segmented_cot'][:step_idx]
    step_prefix = '\n'.join([cot_prefix] + prev_steps) if prev_steps else cot_prefix
    step_prob = completion_probabilities(model, tokenizer, step_prefix, [target['cot']])

    completion, probs, pred = answer_probabilities(model, tokenizer, DH, target['raw_instance'])
    spec_preds, spec_probs = compute_specificity(model, tokenizer, DH, specificity_split)
    new_cot = complete(model, tokenizer, cot_prefix)
    new_cot_probs, _ = generation_fixed_cot(model, tokenizer, DH, target['raw_instance'], new_cot)

    return {
        'completion': completion,
        'probs': probs.tolist(),
        'prediction': pred,
        'target_cot_step': unlearned_step,
        'target_cot_step_prefix': step_prefix,
        'specificity_preds': spec_preds,
        'specificity_probs': spec_probs,
        'new_cot': new_cot,
        'new_cot_probs': new_cot_probs.tolist(),
        'cot_prob': cot_prob.detach().cpu().float().numpy().tolist(),
        'cot_step_prob': step_prob.detach().cpu().float().numpy().tolist(),
    }


def unlearn_single(model_id, tokenizer, config, target, step_idx,
                   cots_train, cots_verify, dh, instance_idx):
    import os as _os
    _os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

    # Trainable model: bfloat16 with gradient checkpointing to save activation memory
    model = CLM.from_pretrained(model_id, torch_dtype=torch.bfloat16,
                                trust_remote_code=True, device_map='auto')
    model.gradient_checkpointing_enable()

    # Oracle model: load in 8-bit (inference only — never trains, saves ~14GB)
    oracle_model = CLM.from_pretrained(model_id,
                                       load_in_8bit=True,
                                       trust_remote_code=True,
                                       device_map='auto')
    oracle_model.eval()
    for p in oracle_model.parameters():
        p.requires_grad = False

    device = next(model.parameters()).device
    collator = FRCollator(tokenizer, device=device)

    dataset = cot_to_otfd(target, cots_train, tokenizer,
                          strategy=config.strategy, stepwise=config.stepwise,
                          step_idx=step_idx, pos=config.pos)
    NT = dataset.num_targets()
    print(f'  Num targets: {NT} | Step: {target["segmented_cot"][step_idx][:60]}...')

    if NT <= 2:
        print('  Too few targets, skipping.')
        del model, oracle_model
        gc.collect(); torch.cuda.empty_cache()
        return {'unlearning_results': None}

    max_steps = config.epochs * len(dataset)
    train_dl = DataLoader(dataset, batch_size=1, collate_fn=collator, shuffle=True)

    if config.ff2:
        for name, param in model.named_parameters():
            param.requires_grad = 'mlp.down_proj.weight' in name

    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr)
    scheduler = get_linear_schedule_with_warmup(optimizer, 0, max_steps)

    results = {}
    results[0] = evaluate_instance(model, tokenizer, dh, target, cots_verify, step_idx)

    for epoch in range(config.epochs):
        model.train()
        optimizer.zero_grad()
        for batch in train_dl:
            loss = compute_loss(model, oracle_model, batch, loss_type=config.method)
            loss.backward()
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        results[epoch + 1] = evaluate_instance(model, tokenizer, dh, target, cots_verify, step_idx)
        print(f'  Epoch {epoch+1}/{config.epochs} done.')

    del collator, train_dl, dataset, scheduler, optimizer, model, oracle_model
    gc.collect(); torch.cuda.empty_cache()
    return {'unlearning_results': results}


def load_ids(fin, stepwise=False):
    ids = set()
    if os.path.exists(fin):
        with open(fin) as f:
            for line in f:
                d = json.loads(line)
                id_ = d['question']
                if stepwise:
                    id_ = f"{id_}_{d['step_idx']}"
                ids.add(id_)
    return ids


def store(instance_info, fout):
    with open(fout, 'a') as f:
        f.write(json.dumps(instance_info) + '\n')


print('✅ Core functions defined.')

## 7. Run Unlearning Experiment 🚀

**Run this cell twice** — once with `ACTIVE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.2'` and once with `ACTIVE_MODEL = 'BioMistral/BioMistral-7B'`.

Results are saved separately per model so you can compare them in Section 9.

In [ ]:
from tqdm.notebook import tqdm

set_random_seed(args.seed)

model_id = args.model_name
tokenizer = TOK.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Load MedQA handler
DH = DATASETS['medqa']

# Generate or load CoTs for this model on MedQA
print(f'\n📝 Loading/generating CoTs for {model_id} on MedQA...')
cot_data = load_or_generate_dataset_cots(
    model_id=model_id, tokenizer=tokenizer,
    dataset_id='medqa', force_generate=args.new_cot,
    sentencize=args.strategy == 'sentencize',
    temperature=args.temperature, seed=args.seed,
    atomic=args.atomic
)

random.shuffle(cot_data)
N_verify = 20
cots_train, cots_verify = cot_data[:-N_verify], cot_data[-N_verify:]

# Results directory — separate per model
short_model = model_id.split('/')[-1].replace('-', '_').lower()
resdir = f'colab_results/medqa/{short_model}/'
os.makedirs(resdir, exist_ok=True)

logfile_name = (f"{args.method}_{args.strategy}_s={args.stepwise}"
                f"_lr={args.lr}_rs={args.seed}"
                f"_pos={args.pos}_ff2={args.ff2}.out")

ids = load_ids(resdir + logfile_name, stepwise=args.stepwise)
print(f'🔄 Resuming from {len(ids)} previously processed instances.')
print(f'📂 Results → {resdir}{logfile_name}')
print(f'🎯 Processing up to {args.max_instances} instances\n')

all_results = []

for idx, target in enumerate(tqdm(cots_train[:args.max_instances], desc='Unlearning instances')):
    n_steps = len(target['segmented_cot']) if args.stepwise else 1

    for step_idx in range(n_steps):
        check_id = target['id']
        if args.stepwise:
            check_id = f"{check_id}_{step_idx}"
        if check_id in ids:
            continue

        print(f"\n{'='*60}")
        print(f'Instance {idx+1}/{args.max_instances}, Step {step_idx+1}/{n_steps}')
        print(f'Q: {target["question"][:100]}...')
        print(f"{'='*60}")

        instance_info = {
            'id': target['id'],
            'question': target['question'],
            'step_idx': step_idx,
            'options': target['options'],
            'correct': target['correct_letter'],
            'initial_cot': target['cot'],
            'initial_cot_probs': target['cot_probs'],
            'initial_probs': target['nocot_probs'],
            'prediction': int(np.argmax(target['nocot_probs'])),
            'cot_prediction': int(np.argmax(target['cot_probs'])),
            'cot_step': target['segmented_cot'][step_idx],
            'segmented_cot': target['segmented_cot'],
            'model': model_id,
        }

        return_dict = unlearn_single(
            model_id, tokenizer, args, target, step_idx,
            cots_train, cots_verify, DH, idx
        )

        results = return_dict['unlearning_results']
        if results is None:
            continue

        instance_info['unlearning_results'] = results
        store(instance_info, resdir + logfile_name)
        all_results.append(instance_info)

        initial_pred = instance_info['cot_prediction']
        final_pred = results[args.epochs]['prediction']
        flipped = '🔄 FLIPPED' if initial_pred != final_pred else '✓ Same'
        print(f'  → {LETTERS[initial_pred]} → {LETTERS[final_pred]} {flipped}')
        memory_stats()

print(f"\n{'='*60}")
print(f'✅ Done! Processed {len(all_results)} instances.')
print(f'📂 Results saved to: {resdir}{logfile_name}')
print(f"{'='*60}")

## 8. Per-Model Statistics

In [ ]:
# Fix integer keys from in-memory results
for r in all_results:
    if 'unlearning_results' in r:
        ur = r['unlearning_results']
        if any(isinstance(k, int) for k in ur.keys()):
            r['unlearning_results'] = {str(k): v for k, v in ur.items()}

from stats import make_stats

if all_results:
    stats = make_stats(all_results)
    print(f'\n📊 Results for: {args.model_name}')
    print('=' * 50)
    print(f'  Instances processed : {stats["n_instances"]}')
    print(f'  CoT steps processed : {stats["n_cot_steps"]}')
    print(f'  FF-HARD (% flipped) : {stats["faithfulness"]:.2f}%')
    print(f'  Efficacy            : {stats["efficacy"]:.2f}%')
    print(f'  Specificity         : {stats["specificity"]:.2f}%')
    print('=' * 50)
else:
    print('⚠️ No results in memory. Loading from disk...')
    result_file = resdir + logfile_name
    if os.path.exists(result_file):
        all_results = []
        with open(result_file) as f:
            for line in f:
                all_results.append(json.loads(line))
        stats = make_stats(all_results)
        print(f'  FF-HARD: {stats["faithfulness"]:.2f}%')
        print(f'  Efficacy: {stats["efficacy"]:.2f}%')
        print(f'  Specificity: {stats["specificity"]:.2f}%')

## 9. Comparative Analysis: BioMistral vs Mistral-7B

Run this cell **after both models have been run**. It loads both result files and computes:
- FF-HARD comparison (prediction flip rate)
- FF-SOFT comparison (average probability drop)
- Bootstrap confidence intervals (α=0.05)
- Bar chart visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json, os

# ---- Load both result files ----
BASE_DIR = 'colab_results/medqa'

MODEL_FILES = {
    'Mistral-7B (General)': os.path.join(
        BASE_DIR, 'mistral_7b_instruct_v0.2',
        'npo_KL_sentencize_s=True_lr=3e-05_rs=1001_pos=True_ff2=True.out'
    ),
    'BioMistral-7B (Medical)': os.path.join(
        BASE_DIR, 'biomistral_7b',
        'npo_KL_sentencize_s=True_lr=3e-05_rs=1001_pos=True_ff2=True.out'
    ),
}

def load_results(path):
    results = []
    if not os.path.exists(path):
        print(f'⚠️ File not found: {path}')
        return results
    with open(path) as f:
        for line in f:
            results.append(json.loads(line))
    return results


def compute_ff_hard(results, final_epoch='5'):
    """FF-HARD: fraction of steps where prediction flipped after unlearning."""
    flips = []
    for r in results:
        ur = r.get('unlearning_results', {})
        if not ur:
            continue
        initial = r['cot_prediction']
        final_key = str(final_epoch)
        if final_key not in ur:
            final_key = str(max(int(k) for k in ur.keys()))
        final = ur[final_key]['prediction']
        flips.append(int(initial != final))
    return np.array(flips)


def compute_ff_soft(results, final_epoch='5'):
    """FF-SOFT: probability mass shifted away from initial prediction."""
    shifts = []
    for r in results:
        ur = r.get('unlearning_results', {})
        if not ur:
            continue
        initial_pred = r['cot_prediction']
        initial_prob = ur['0']['probs'][initial_pred]
        final_key = str(final_epoch)
        if final_key not in ur:
            final_key = str(max(int(k) for k in ur.keys()))
        final_prob = ur[final_key]['probs'][initial_pred]
        shifts.append(initial_prob - final_prob)  # positive = mass moved away
    return np.array(shifts)


def bootstrap_ci(data, n_boot=10000, alpha=0.05, stat_fn=np.mean):
    """Bootstrap confidence interval for a statistic."""
    rng = np.random.default_rng(42)
    boot_stats = [stat_fn(rng.choice(data, size=len(data), replace=True)) for _ in range(n_boot)]
    lo = np.percentile(boot_stats, 100 * alpha / 2)
    hi = np.percentile(boot_stats, 100 * (1 - alpha / 2))
    return stat_fn(data), lo, hi


# ---- Compute metrics ----
print('\n📊 Comparative Analysis: BioMistral-7B vs Mistral-7B on MedQA')
print('=' * 65)
print(f'{"Model":<30} {"FF-HARD":>10} {"95% CI":>20} {"FF-SOFT":>10} {"95% CI":>20}')
print('-' * 65)

summary = {}
for label, path in MODEL_FILES.items():
    results = load_results(path)
    if not results:
        print(f'{label:<30}  (no data yet)')
        continue

    ff_hard = compute_ff_hard(results)
    ff_soft = compute_ff_soft(results)

    hard_mean, hard_lo, hard_hi = bootstrap_ci(ff_hard)
    soft_mean, soft_lo, soft_hi = bootstrap_ci(ff_soft)

    summary[label] = {
        'ff_hard': ff_hard, 'ff_soft': ff_soft,
        'hard_mean': hard_mean, 'hard_lo': hard_lo, 'hard_hi': hard_hi,
        'soft_mean': soft_mean, 'soft_lo': soft_lo, 'soft_hi': soft_hi,
        'n': len(ff_hard)
    }

    print(f'{label:<30} {hard_mean*100:>9.1f}% [{hard_lo*100:.1f}%, {hard_hi*100:.1f}%]  '
          f'{soft_mean:>9.4f} [{soft_lo:.4f}, {soft_hi:.4f}]')

print('=' * 65)

In [ ]:
# ---- Permutation test for FF-HARD difference ----
if len(summary) == 2:
    labels = list(summary.keys())
    a = summary[labels[0]]['ff_hard']
    b = summary[labels[1]]['ff_hard']
    observed_diff = np.mean(b) - np.mean(a)  # BioMistral - Mistral

    rng = np.random.default_rng(42)
    combined = np.concatenate([a, b])
    n_a = len(a)
    perm_diffs = []
    for _ in range(10000):
        perm = rng.permutation(combined)
        perm_diffs.append(np.mean(perm[n_a:]) - np.mean(perm[:n_a]))

    p_value = np.mean(np.abs(perm_diffs) >= np.abs(observed_diff))

    print(f'\n🔬 Permutation Test (FF-HARD difference)')
    print(f'   Observed difference (BioMistral - Mistral): {observed_diff*100:+.2f}%')
    print(f'   p-value (two-tailed): {p_value:.4f}')
    if p_value < 0.05:
        print(f'   ✅ Statistically significant at α=0.05')
    else:
        print(f'   ⚪ Not statistically significant at α=0.05 (null hypothesis not rejected)')
else:
    print('⚠️ Need both model results to run permutation test.')

In [ ]:
# ---- Visualization ----
if len(summary) >= 1:
    import matplotlib.pyplot as plt
    import numpy as np

    labels = list(summary.keys())
    colors = ['#2196F3', '#E91E63']

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # FF-HARD bar chart
    hard_means = [summary[l]['hard_mean'] * 100 for l in labels]
    hard_errs = [
        [summary[l]['hard_mean'] * 100 - summary[l]['hard_lo'] * 100 for l in labels],
        [summary[l]['hard_hi'] * 100 - summary[l]['hard_mean'] * 100 for l in labels]
    ]
    bars = axes[0].bar(labels, hard_means, color=colors[:len(labels)],
                       yerr=hard_errs, capsize=6, alpha=0.85, edgecolor='black')
    axes[0].set_ylabel('FF-HARD (%)', fontsize=12)
    axes[0].set_title('Prediction Flip Rate (FF-HARD)\nMedQA', fontsize=12)
    axes[0].set_ylim(0, 100)
    axes[0].tick_params(axis='x', labelsize=9)
    for bar, val in zip(bars, hard_means):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                     f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

    # FF-SOFT bar chart
    soft_means = [summary[l]['soft_mean'] for l in labels]
    soft_errs = [
        [summary[l]['soft_mean'] - summary[l]['soft_lo'] for l in labels],
        [summary[l]['soft_hi'] - summary[l]['soft_mean'] for l in labels]
    ]
    bars2 = axes[1].bar(labels, soft_means, color=colors[:len(labels)],
                        yerr=soft_errs, capsize=6, alpha=0.85, edgecolor='black')
    axes[1].set_ylabel('FF-SOFT (avg prob shift)', fontsize=12)
    axes[1].set_title('Probability Mass Shift (FF-SOFT)\nMedQA', fontsize=12)
    axes[1].tick_params(axis='x', labelsize=9)
    for bar, val in zip(bars2, soft_means):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                     f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.suptitle('BioMistral-7B vs Mistral-7B: Parametric Faithfulness on MedQA',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('medqa_faithfulness_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Plot saved as medqa_faithfulness_comparison.png')

## 10. Download Results

In [ ]:
try:
    from google.colab import files
    result_file = resdir + logfile_name
    if os.path.exists(result_file):
        files.download(result_file)
        print(f'✅ Downloaded: {result_file}')
    if os.path.exists('medqa_faithfulness_comparison.png'):
        files.download('medqa_faithfulness_comparison.png')
        print('✅ Downloaded: medqa_faithfulness_comparison.png')
except ImportError:
    print(f'📁 Results at: {os.path.abspath(resdir + logfile_name)}')